# Добор размеченных отзывов из HDBSCAN-кластеров

Этот ноутбук добирает дополнительные размеченные примеры к уже имеющемуся датасету.

Логика:
1. Берем результат HDBSCAN: `hdbscan_clusters.parquet`.
2. Берем LLM-анализ кластеров: `cluster_llm_analysis.csv`.
3. Выбираем кластеры, которые могут содержать нужные классы из схемы разметки.
4. Проходим по отзывам внутри выбранных кластеров.
5. Каждый отзыв отдельно размечаем через LLM тем же prompt, что и раньше.
6. Сохраняем каждый валидный пример сразу на диск.
7. Лимита `50` на класс нет: собираем столько, сколько получится из выбранных кластеров.

Важно: описание кластера используется только как фильтр. Финальная разметка каждого отзыва делается отдельно через LLM.

## 1. Подключение Google Drive и установка библиотек

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip -q install groq pandas tqdm pyarrow openpyxl

## 2. Импорты

In [ ]:
import os
import re
import json
import time
import hashlib
from pathlib import Path
from getpass import getpass

import pandas as pd
from tqdm.auto import tqdm
from groq import Groq

## 3. Основные настройки

Проверь только этот блок перед запуском.

In [ ]:
# =========================
# Пути проекта
# =========================
BASE_DIR = Path("/content/drive/MyDrive/MLops_project/project")
DATA_DIR = BASE_DIR / "data"

# Папка с результатами HDBSCAN и LLM-анализа кластеров
RESULT_DIR = DATA_DIR / "hdbscan_results_raw_sample_200k"

# Входные файлы из analyze_HDBSCAN
CLUSTERS_PATH = RESULT_DIR / "hdbscan_clusters.parquet"
CLUSTER_ANALYSIS_CSV_PATH = RESULT_DIR / "cluster_llm_analysis.csv"

# Если уже был сохранен объединенный файл из analyze_HDBSCAN, можно использовать его.
# Ноутбук умеет работать и без него, через merge CLUSTERS_PATH + CLUSTER_ANALYSIS_CSV_PATH.
ENRICHED_CLUSTERS_PATH = RESULT_DIR / "hdbscan_clusters_with_llm_analysis.parquet"

# Старый датасет с разметкой, к которому добираем дополнительные примеры.
# Если такого файла нет, ноутбук просто начнет новый добор.
EXISTING_LABELED_CSV_PATH = (
    BASE_DIR / "data" / "labeled" / "wb_feedbacks_llm_labeled_from_sample" /
    "balanced_50_per_class_random_chunks.csv"
)

# Куда сохранять новую разметку из кластеров
LABELED_DIR = BASE_DIR / "data" / "labeled" / "wb_feedbacks_llm_labeled_from_clusters"
LABELED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_BASENAME = "extra_from_hdbscan_candidate_clusters"
OUTPUT_CSV_PATH = LABELED_DIR / f"{OUTPUT_BASENAME}.csv"
OUTPUT_PARQUET_PATH = LABELED_DIR / f"{OUTPUT_BASENAME}.parquet"
FINAL_CSV_PATH = LABELED_DIR / f"{OUTPUT_BASENAME}_final.csv"
STATUS_CSV_PATH = LABELED_DIR / f"{OUTPUT_BASENAME}_class_status.csv"
RUN_STATS_JSON_PATH = LABELED_DIR / f"{OUTPUT_BASENAME}_run_stats.json"

# =========================
# Колонки
# =========================
TEXT_COL = "text"
CLUSTER_COL = "cluster_id"
PROB_COL = "cluster_probability"

# =========================
# Режим выбора кластеров
# =========================
# По умолчанию пытаемся добирать все разрешенные классы.
# Если нужно добирать только проблемные классы, ниже можно убрать "Положительный отзыв" и "Нейтральный / информационный отзыв".
TARGET_CLUSTER_LABELS = None  # None = использовать все ALLOWED_LABELS

# Минимальная уверенность LLM-анализа кластера.
# 0.0 = брать все кластеры, где есть matched_labels.
MIN_CLUSTER_ANALYSIS_CONFIDENCE = 0.0

# Брать ли кластеры без matched_labels, но с негативным / смешанным типом.
# Это полезно, если LLM описал кластер как problem/mixed, но не сопоставил с конкретным классом.
INCLUDE_NEGATIVE_OR_MIXED_WITHOUT_MATCHED_LABELS = False

# Ограничители для тестового запуска.
# Для полного запуска оставь None.
MAX_CLUSTERS_TO_PROCESS = None
MAX_REVIEWS_PER_CLUSTER = None
MAX_REVIEWS_TO_SEND_TO_MODEL = None

# Если True, внутри каждого кластера отзывы будут идти от самой высокой cluster_probability к меньшей.
SORT_REVIEWS_BY_CLUSTER_PROBABILITY = True

# Дедупликация
DEDUP_BY_TEXT = True

# Если True, удалит старые выходные файлы этого ноутбука и начнет заново.
# Обычно оставляй False, чтобы можно было продолжить после остановки.
RESET_OUTPUT_FILES_ON_START = False
RESUME_FROM_EXISTING_OUTPUT = True

# =========================
# API и модель
# =========================
API_KEY_ENV_NAME = "GROQ_API_KEY"
MODEL_NAME = "qwen/qwen3-32b"
TEMPERATURE = 0
MAX_RETRIES = 3
RETRY_BASE_SLEEP_SECONDS = 2
REQUEST_SLEEP_SECONDS = 0.0

## 4. Разрешенные классы

In [ ]:
ALLOWED_LABELS = [
    "Брак / дефект товара",
    "Низкое качество материала",
    "Проблема с размером / посадкой",
    "Несоответствие описанию",
    "Несоответствие ожиданиям / эффекту",
    "Проблема с комплектацией",
    "Проблема с упаковкой",
    "Проблема доставки / получения",
    "Цена / ценность",
    "Подделка / оригинальность",
    "Положительный отзыв",
    "Нейтральный / информационный отзыв",
    "Другая проблема",
]

if TARGET_CLUSTER_LABELS is None:
    TARGET_CLUSTER_LABELS = ALLOWED_LABELS.copy()
else:
    unknown = [x for x in TARGET_CLUSTER_LABELS if x not in ALLOWED_LABELS]
    if unknown:
        raise ValueError(f"В TARGET_CLUSTER_LABELS есть неизвестные классы: {unknown}")

PROBLEM_LABELS = [
    label for label in ALLOWED_LABELS
    if label not in ["Положительный отзыв", "Нейтральный / информационный отзыв", "Другая проблема"]
]

print("Всего разрешенных классов:", len(ALLOWED_LABELS))
print("Классы для поиска в кластерах:")
for label in TARGET_CLUSTER_LABELS:
    print("-", label)

## 5. Prompt для разметки одного отзыва

Оставлен тот же принцип, что во втором ноутбуке: каждый отзыв размечается отдельно и возвращается строго JSON.

In [ ]:
USER_PROMPT = """
/no_think

Ты выполняешь разметку отзывов покупателей о товарах на маркетплейсе.

Твоя задача — определить, к каким классам относится отзыв. Это multi-label классификация: один отзыв может относиться к одному или нескольким классам.

Важно:

* Размечай только по тексту отзыва.
* Не додумывай факты, которых нет в отзыве.
* Не используй знания о товаре, бренде или маркетплейсе, если они не указаны в тексте.
* Выбирай класс только если в отзыве есть явный или достаточно понятный признак этого класса.
* Нельзя придумывать новые классы.
* Названия классов должны полностью совпадать со списком разрешенных классов.

Разрешенные классы:

1. "Брак / дефект товара"
   Выбирай, если товар сломан, поврежден, не работает, работает неправильно, быстро сломался, имеет трещины, сколы, заводской дефект или явную неисправность.
   Не выбирай этот класс, если товар просто сделан из плохого материала, но не сломан.

2. "Низкое качество материала"
   Выбирай, если покупатель жалуется на плохой материал, дешевый пластик, плохую ткань, кривые швы, хлипкость, слабую сборку, неприятный материал, быструю изнашиваемость.
   Не выбирай этот класс только из-за высокой цены или поломки.

3. "Проблема с размером / посадкой"
   Выбирай, если товар не подошел по размеру, форме, посадке или совместимости: маломерит, большемерит, неудобно сидит, не подошел к устройству, оказался меньше или больше ожидаемого.
   Если в отзыве явно сказано, что в карточке были неверные размеры или характеристики, можно дополнительно выбрать "Несоответствие описанию".

4. "Несоответствие описанию"
   Выбирай, если товар отличается от карточки товара, фото, описания, характеристик или заявленных свойств: другой цвет, другой материал, другая модель, не те характеристики, описание вводит в заблуждение.
   Не выбирай этот класс, если покупатель просто недоволен качеством, но не сравнивает товар с описанием.

5. "Несоответствие ожиданиям / эффекту"
   Выбирай, если товар формально пришел и не сломан, но не выполняет ожидаемую функцию или не дает результата: средство не помогает, устройство слабое, товар бесполезен, эффект отсутствует, результат хуже ожидаемого.
   Не выбирай этот класс, если товар именно неисправен или сломан.

6. "Проблема с комплектацией"
   Выбирай, если в заказе не хватает деталей, аксессуаров, инструкции, кабеля, насадок, креплений или других элементов комплекта; пришла только часть набора; комплект неполный.

7. "Проблема с упаковкой"
   Выбирай, если жалоба относится к упаковке: коробка мятая, упаковка вскрыта, порвана, плохая защита, товар был плохо упакован.
   Если поврежден только внешний вид упаковки, но товар целый, выбирай только этот класс.
   Если сам товар сломан или поврежден, дополнительно выбирай "Брак / дефект товара", если это явно следует из текста.

8. "Проблема доставки / получения"
   Выбирай, если проблема связана с доставкой или получением: долго ехал, задержали, перенесли доставку, пришел не туда, проблемы с пунктом выдачи, курьером, логистикой, заказ потеряли.
   Не выбирай этот класс только из-за мятой коробки — для этого есть "Проблема с упаковкой".

9. "Цена / ценность"
   Выбирай, если покупатель пишет, что товар не стоит своих денег, цена завышена, качество не соответствует цене, можно найти дешевле, за эти деньги ожидал большего.
   Не выбирай этот класс, если в отзыве просто плохое качество без упоминания цены или ценности.

10. "Подделка / оригинальность"
    Выбирай, если покупатель сомневается в подлинности товара или пишет, что товар похож на подделку, неоригинальный, не фирменный, отличается от оригинала, вызывает сомнения бренд, маркировка или упаковка.

11. "Положительный отзыв"
    Выбирай, если покупатель явно доволен товаром: товар понравился, все хорошо, рекомендую, качество устраивает, покупкой доволен, товар оправдал ожидания.
    Этот класс можно сочетать с проблемным классом, если отзыв смешанный: есть и похвала, и конкретная жалоба.
    Не выбирай этот класс, если в отзыве нет явной положительной оценки.

12. "Нейтральный / информационный отзыв"
    Выбирай, если отзыв не содержит явной положительной или отрицательной оценки, а только сообщает факт: получил, пока не использовал, дополню позже, купил в подарок, не открывал, пока ничего сказать не могу.
    Этот класс нельзя сочетать с другими классами.
    Не выбирай этот класс, если есть явная похвала или жалоба.

13. "Другая проблема"
    Выбирай, если в отзыве есть негативная проблема, но ее нельзя уверенно отнести ни к одному из основных классов.
    Этот класс используй только как запасной вариант для непонятной негативной жалобы.
    Не выбирай "Другая проблема", если можно выбрать более конкретный класс.
    Обычно этот класс не нужно сочетать с другими проблемными классами.

Правила выбора классов:

1. Один отзыв может иметь несколько классов.
2. Если отзыв содержит и похвалу, и жалобу, выбери "Положительный отзыв" и соответствующий проблемный класс.
3. "Нейтральный / информационный отзыв" выбирай только тогда, когда нет ни похвалы, ни жалобы.
4. "Нейтральный / информационный отзыв" не должен идти вместе с другими классами.
5. "Другая проблема" выбирай только если есть негатив, но нет подходящего конкретного класса.
6. Если проблема относится к самому товару, выбирай товарный класс: брак, качество, размер, комплектация, описание, эффект, цена, оригинальность.
7. Если проблема относится к процессу получения заказа, выбирай доставку или упаковку.
8. Не добавляй слишком много классов. Каждый выбранный класс должен подтверждаться фрагментом отзыва.
9. При сомнении между конкретным классом и "Другая проблема" выбирай конкретный класс.
10. При сомнении между "Положительный отзыв" и "Нейтральный / информационный отзыв" выбирай "Положительный отзыв" только при явной похвале.
11. Если отзыв очень короткий, например "норм", "ок", "пришло", "получил", и нет понятной оценки, выбирай "Нейтральный / информационный отзыв".
12. Если отзыв короткий, но содержит явную оценку, например "отлично", "супер", "ужас", классифицируй по этой оценке.

Как выставлять confidence:

* 0.90–1.00: класс очевиден, есть прямые слова из отзыва.
* 0.70–0.89: класс достаточно понятен, но формулировка не полностью прямая.
* 0.50–0.69: отзыв неоднозначный, но выбранный класс наиболее вероятен.
* ниже 0.50: используй только если отзыв очень неясный.

Формат ответа:

Верни только валидный JSON-объект без markdown и без текста вне JSON.

JSON должен иметь ровно такие поля:
{
"labels": ["один или несколько классов из разрешенного списка"],
"confidence": 0.0,
"evidence": "короткое объяснение на русском, какие слова из отзыва подтверждают выбор"
}
""".strip()

## 6. API-ключ и клиент

In [ ]:
if API_KEY_ENV_NAME not in os.environ or not os.environ[API_KEY_ENV_NAME]:
    os.environ[API_KEY_ENV_NAME] = getpass(f"Вставь {API_KEY_ENV_NAME}: ")

client = Groq(api_key=os.environ[API_KEY_ENV_NAME])

print("MODEL_NAME:", MODEL_NAME)
print("OUTPUT_CSV_PATH:", OUTPUT_CSV_PATH)

## 7. Загрузка кластеров и анализа кластеров

In [ ]:
if not CLUSTERS_PATH.exists():
    raise FileNotFoundError(
        f"Не найден файл с кластерами: {CLUSTERS_PATH}\n"
        "Сначала запусти ноутбук с HDBSCAN, который создает hdbscan_clusters.parquet."
    )

if not CLUSTER_ANALYSIS_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Не найден файл анализа кластеров: {CLUSTER_ANALYSIS_CSV_PATH}\n"
        "Сначала запусти analyze_HDBSCAN, который создает cluster_llm_analysis.csv."
    )

clusters_df = pd.read_parquet(CLUSTERS_PATH)
cluster_analysis_df = pd.read_csv(CLUSTER_ANALYSIS_CSV_PATH)

required_cluster_cols = [TEXT_COL, CLUSTER_COL]
missing_cluster_cols = [col for col in required_cluster_cols if col not in clusters_df.columns]
if missing_cluster_cols:
    raise ValueError(f"В CLUSTERS_PATH не хватает колонок: {missing_cluster_cols}. Колонки: {list(clusters_df.columns)}")

if CLUSTER_COL not in cluster_analysis_df.columns:
    raise ValueError(f"В cluster_llm_analysis.csv нет колонки {CLUSTER_COL}. Колонки: {list(cluster_analysis_df.columns)}")

clusters_df = clusters_df.copy()
clusters_df[TEXT_COL] = clusters_df[TEXT_COL].astype(str)
clusters_df[CLUSTER_COL] = clusters_df[CLUSTER_COL].astype(int)

if PROB_COL in clusters_df.columns:
    clusters_df[PROB_COL] = pd.to_numeric(clusters_df[PROB_COL], errors="coerce").fillna(0.0)
else:
    clusters_df[PROB_COL] = 0.0

cluster_analysis_df = cluster_analysis_df.copy()
cluster_analysis_df[CLUSTER_COL] = cluster_analysis_df[CLUSTER_COL].astype(int)

print("clusters_df:", clusters_df.shape)
print("cluster_analysis_df:", cluster_analysis_df.shape)
print("Колонки clusters_df:", list(clusters_df.columns))
print("Колонки cluster_analysis_df:", list(cluster_analysis_df.columns))

display(clusters_df.head())
display(cluster_analysis_df.head())

## 8. Выбор кластеров-кандидатов

Кластер выбирается, если в `matched_labels_str` или `matched_labels` есть хотя бы один нужный класс.

In [ ]:
def parse_matched_labels(value):
    """
    Приводит matched_labels / matched_labels_str к списку разрешенных классов.
    Поддерживает:
    - строку вида "класс 1; класс 2";
    - JSON-строку списка;
    - настоящий list.
    """
    if isinstance(value, list):
        raw = value
    elif pd.isna(value):
        raw = []
    else:
        text = str(value).strip()
        if not text:
            raw = []
        else:
            try:
                parsed = json.loads(text)
                raw = parsed if isinstance(parsed, list) else [text]
            except Exception:
                raw = re.split(r"\s*;\s*|\s*,\s*", text)

    labels = []
    for item in raw:
        label = str(item).strip().strip('"').strip("'")
        if label in ALLOWED_LABELS and label not in labels:
            labels.append(label)
    return labels


def choose_candidate_clusters(cluster_analysis_df):
    df = cluster_analysis_df.copy()

    if "matched_labels" in df.columns:
        df["matched_labels_list"] = df["matched_labels"].apply(parse_matched_labels)
    elif "matched_labels_str" in df.columns:
        df["matched_labels_list"] = df["matched_labels_str"].apply(parse_matched_labels)
    else:
        df["matched_labels_list"] = [[] for _ in range(len(df))]

    if "matched_labels_str" not in df.columns:
        df["matched_labels_str"] = df["matched_labels_list"].apply(lambda xs: "; ".join(xs))

    if "confidence" in df.columns:
        df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce").fillna(0.0)
    else:
        df["confidence"] = 0.0

    target_set = set(TARGET_CLUSTER_LABELS)

    df["target_matched_labels"] = df["matched_labels_list"].apply(
        lambda xs: [x for x in xs if x in target_set]
    )
    df["has_target_label"] = df["target_matched_labels"].apply(lambda xs: len(xs) > 0)

    base_mask = (
        df["has_target_label"]
        & (df["confidence"] >= MIN_CLUSTER_ANALYSIS_CONFIDENCE)
        & (df[CLUSTER_COL] != -1)
    )

    if INCLUDE_NEGATIVE_OR_MIXED_WITHOUT_MATCHED_LABELS and "positive_or_negative" in df.columns:
        pon = df["positive_or_negative"].astype(str).str.lower()
        fallback_mask = (
            pon.isin(["negative", "mixed"])
            & (df["confidence"] >= MIN_CLUSTER_ANALYSIS_CONFIDENCE)
            & (df[CLUSTER_COL] != -1)
        )
        mask = base_mask | fallback_mask
    else:
        mask = base_mask

    selected = df[mask].copy()

    if "cluster_size" in selected.columns:
        selected["cluster_size"] = pd.to_numeric(selected["cluster_size"], errors="coerce").fillna(0).astype(int)
        selected = selected.sort_values(["cluster_size", "confidence"], ascending=[False, False])
    else:
        sizes = clusters_df.groupby(CLUSTER_COL).size().rename("cluster_size").reset_index()
        selected = selected.merge(sizes, on=CLUSTER_COL, how="left")
        selected = selected.sort_values(["cluster_size", "confidence"], ascending=[False, False])

    if MAX_CLUSTERS_TO_PROCESS is not None:
        selected = selected.head(MAX_CLUSTERS_TO_PROCESS).copy()

    return selected

candidate_clusters_df = choose_candidate_clusters(cluster_analysis_df)
CANDIDATE_CLUSTER_IDS = candidate_clusters_df[CLUSTER_COL].astype(int).tolist()

print("Выбрано кластеров-кандидатов:", len(CANDIDATE_CLUSTER_IDS))
print("Всего отзывов в этих кластерах:", clusters_df[clusters_df[CLUSTER_COL].isin(CANDIDATE_CLUSTER_IDS)].shape[0])

display_cols = [
    CLUSTER_COL,
    "cluster_size",
    "cluster_title",
    "cluster_feature",
    "main_theme",
    "matched_labels_str",
    "target_matched_labels",
    "positive_or_negative",
    "confidence",
    "evidence",
]
display_cols = [col for col in display_cols if col in candidate_clusters_df.columns]

display(candidate_clusters_df[display_cols].head(50))

## 9. Быстрая проверка примеров из выбранных кластеров

Перед массовой разметкой полезно посмотреть, действительно ли выбранные кластеры похожи на нужные.

In [ ]:
N_CLUSTERS_TO_PREVIEW = min(5, len(CANDIDATE_CLUSTER_IDS))
N_EXAMPLES_PER_CLUSTER_PREVIEW = 5

for cluster_id in CANDIDATE_CLUSTER_IDS[:N_CLUSTERS_TO_PREVIEW]:
    print("=" * 100)
    print("cluster_id:", cluster_id)

    meta = candidate_clusters_df[candidate_clusters_df[CLUSTER_COL] == cluster_id]
    if not meta.empty:
        cols = ["cluster_title", "matched_labels_str", "positive_or_negative", "confidence", "cluster_feature"]
        cols = [c for c in cols if c in meta.columns]
        display(meta[cols].head(1))

    sample = clusters_df[clusters_df[CLUSTER_COL] == cluster_id].copy()
    sample = sample.sort_values(PROB_COL, ascending=False).head(N_EXAMPLES_PER_CLUSTER_PREVIEW)
    display(sample[[TEXT_COL, PROB_COL]].head(N_EXAMPLES_PER_CLUSTER_PREVIEW))

## 10. Функции разметки одного отзыва

In [ ]:
def safe_json_loads(content):
    content = str(content).strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        start = content.find("{")
        end = content.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(content[start:end + 1])
        raise


def build_full_prompt(review_text):
    allowed_labels_text = "\n".join(f'- "{label}"' for label in ALLOWED_LABELS)
    review_text_json = json.dumps(str(review_text), ensure_ascii=False)

    return f"""
{USER_PROMPT}

Разрешенный список классов, который можно использовать в поле labels:
{allowed_labels_text}

Отзыв для классификации:
{review_text_json}

Верни ответ строго в JSON формате:

{{
  "labels": ["..."],
  "confidence": 0.0,
  "evidence": "короткое объяснение, почему выбраны эти классы"
}}

Жесткие требования к ответу:
- Верни только один JSON-объект.
- Не возвращай markdown.
- Не возвращай текст вне JSON.
- Не добавляй комментарии до или после JSON.
- Поле labels должно быть массивом строк.
- labels может содержать только классы из разрешенного списка.
- Если выбрано "Нейтральный / информационный отзыв", это должен быть единственный класс.
- Если выбрано "Другая проблема", не добавляй ее вместе с конкретным проблемным классом.
- confidence — число от 0 до 1.
- evidence — короткое объяснение на русском.
""".strip()


def normalize_model_response(content):
    data = safe_json_loads(content)

    labels = data.get("labels", [])
    if not isinstance(labels, list):
        labels = []

    normalized_labels = []
    for label in labels:
        label = str(label).strip()
        if label in ALLOWED_LABELS and label not in normalized_labels:
            normalized_labels.append(label)

    # Нейтральный класс не должен сочетаться с другими.
    if "Нейтральный / информационный отзыв" in normalized_labels and len(normalized_labels) > 1:
        normalized_labels = [x for x in normalized_labels if x != "Нейтральный / информационный отзыв"]

    # "Другая проблема" убирается, если есть конкретный проблемный класс.
    if "Другая проблема" in normalized_labels:
        has_concrete_problem = any(label in PROBLEM_LABELS for label in normalized_labels)
        if has_concrete_problem:
            normalized_labels = [x for x in normalized_labels if x != "Другая проблема"]

    if len(normalized_labels) == 0:
        return {
            "labels": [],
            "confidence": 0.0,
            "evidence": "",
            "raw_response": str(content),
            "error": "no_valid_labels",
        }

    try:
        confidence = float(data.get("confidence", 0.0))
    except Exception:
        confidence = 0.0
    confidence = max(0.0, min(1.0, confidence))

    return {
        "labels": normalized_labels,
        "confidence": confidence,
        "evidence": str(data.get("evidence", "")),
        "raw_response": str(content),
        "error": None,
    }


def classify_review(
    review_text,
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
    max_retries=MAX_RETRIES,
):
    full_prompt = build_full_prompt(review_text)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "/no_think\n"
                            "Ты ассистент для строгой разметки данных для обучения ML-классификатора. "
                            "Твоя цель — высокая точность разметки, а не красивое объяснение. "
                            "Отвечай только валидным JSON-объектом. "
                            "Не добавляй markdown, рассуждения, комментарии или текст вне JSON."
                        ),
                    },
                    {
                        "role": "user",
                        "content": full_prompt,
                    },
                ],
                temperature=temperature,
                response_format={"type": "json_object"},
            )

            content = response.choices[0].message.content
            return normalize_model_response(content)

        except Exception as e:
            error_text = str(e)

            if (
                "429" in error_text
                or "rate limit" in error_text.lower()
                or "tokens per day" in error_text.lower()
                or "TPD" in error_text
            ):
                print("\n" + "=" * 80)
                print("ДОСТИГНУТ ЛИМИТ API / RATE LIMIT")
                print(f"attempt: {attempt + 1}/{max_retries}")
                print(error_text)
                print("=" * 80 + "\n")

            if attempt == max_retries - 1:
                return {
                    "labels": [],
                    "confidence": 0.0,
                    "evidence": "",
                    "raw_response": "",
                    "error": error_text,
                }

            time.sleep(RETRY_BASE_SLEEP_SECONDS ** attempt)

## 11. Дедупликация и сохранение результата

In [ ]:
def normalize_text_for_hash(text):
    text = str(text).strip().lower()
    text = " ".join(text.split())
    return text


def text_hash(text):
    return hashlib.md5(normalize_text_for_hash(text).encode("utf-8")).hexdigest()


def parse_labels_str(labels_str):
    if pd.isna(labels_str):
        return []
    return [x.strip() for x in str(labels_str).split(";") if x.strip()]


def load_existing_text_hashes_and_output():
    """
    Загружает:
    - тексты из старого размеченного файла, чтобы не добирать дубликаты;
    - тексты из уже сохраненного результата этого ноутбука, чтобы продолжать после остановки;
    - уже сохраненные строки текущего результата.
    """
    existing_hashes = set()
    already_saved_rows = []

    if DEDUP_BY_TEXT and EXISTING_LABELED_CSV_PATH.exists():
        old_df = pd.read_csv(EXISTING_LABELED_CSV_PATH)
        text_candidates = ["text", "отзыв", TEXT_COL]
        old_text_col = next((c for c in text_candidates if c in old_df.columns), None)
        if old_text_col is not None:
            for text in old_df[old_text_col].dropna().astype(str):
                existing_hashes.add(text_hash(text))
            print("Текстов для дедупликации из старого датасета:", len(existing_hashes))
        else:
            print("Старый файл найден, но текстовая колонка не найдена:", EXISTING_LABELED_CSV_PATH)
    else:
        print("Старый размеченный датасет не найден или DEDUP_BY_TEXT=False:", EXISTING_LABELED_CSV_PATH)

    if RESUME_FROM_EXISTING_OUTPUT and OUTPUT_CSV_PATH.exists():
        current_df = pd.read_csv(OUTPUT_CSV_PATH)
        already_saved_rows = current_df.to_dict("records")
        if TEXT_COL in current_df.columns:
            for text in current_df[TEXT_COL].dropna().astype(str):
                existing_hashes.add(text_hash(text))
        print("Найден текущий результат для продолжения:", OUTPUT_CSV_PATH)
        print("Уже сохранено строк в текущем результате:", len(already_saved_rows))
    else:
        print("Текущий результат не найден. Новый добор начнется с нуля.")

    return existing_hashes, already_saved_rows


def append_example_to_disk(example_row):
    full_row_df = pd.DataFrame([example_row])

    full_row_df.to_csv(
        OUTPUT_CSV_PATH,
        mode="a",
        header=not OUTPUT_CSV_PATH.exists(),
        index=False,
        encoding="utf-8-sig",
    )

    final_row_df = pd.DataFrame([{
        "отзыв": example_row.get(TEXT_COL, ""),
        "labels": example_row.get("labels_str", ""),
        "evidence": example_row.get("evidence", ""),
    }])

    final_row_df.to_csv(
        FINAL_CSV_PATH,
        mode="a",
        header=not FINAL_CSV_PATH.exists(),
        index=False,
        encoding="utf-8-sig",
    )


def save_full_outputs(rows):
    df = pd.DataFrame(rows)
    if not df.empty:
        df.to_parquet(OUTPUT_PARQUET_PATH, index=False)
    return df


def build_class_status_df(rows):
    counts = {label: 0 for label in ALLOWED_LABELS}
    for row in rows:
        labels_counted = row.get("labels_counted", [])
        if isinstance(labels_counted, str):
            labels_counted = parse_labels_str(labels_counted)
        for label in labels_counted:
            if label in counts:
                counts[label] += 1

    status_df = pd.DataFrame([
        {"label": label, "count_in_extra_dataset": counts[label]}
        for label in ALLOWED_LABELS
    ])
    status_df.to_csv(STATUS_CSV_PATH, index=False, encoding="utf-8-sig")
    return status_df

## 12. Основной цикл добора из кластеров

Сохранение происходит после каждого найденного валидного примера.

In [ ]:
def collect_extra_examples_from_candidate_clusters():
    if RESET_OUTPUT_FILES_ON_START:
        for path in [OUTPUT_CSV_PATH, OUTPUT_PARQUET_PATH, FINAL_CSV_PATH, STATUS_CSV_PATH, RUN_STATS_JSON_PATH]:
            if path.exists():
                path.unlink()
        print("Старые выходные файлы удалены.")

    existing_hashes, selected_examples = load_existing_text_hashes_and_output()

    candidate_df = clusters_df[clusters_df[CLUSTER_COL].isin(CANDIDATE_CLUSTER_IDS)].copy()

    # Добавляем метаинформацию о кластере к каждому отзыву.
    meta_cols = [
        CLUSTER_COL,
        "cluster_title",
        "cluster_feature",
        "main_theme",
        "matched_labels_str",
        "positive_or_negative",
        "confidence",
    ]
    meta_cols = [c for c in meta_cols if c in candidate_clusters_df.columns]
    candidate_df = candidate_df.merge(
        candidate_clusters_df[meta_cols].drop_duplicates(subset=[CLUSTER_COL]),
        on=CLUSTER_COL,
        how="left",
        suffixes=("", "_cluster_analysis"),
    )

    # Идем по кластерам в порядке candidate_clusters_df.
    rows_by_cluster = []
    for cluster_id in CANDIDATE_CLUSTER_IDS:
        part = candidate_df[candidate_df[CLUSTER_COL] == cluster_id].copy()
        if SORT_REVIEWS_BY_CLUSTER_PROBABILITY:
            part = part.sort_values(PROB_COL, ascending=False)
        if MAX_REVIEWS_PER_CLUSTER is not None:
            part = part.head(MAX_REVIEWS_PER_CLUSTER)
        rows_by_cluster.append(part)

    if rows_by_cluster:
        candidate_df = pd.concat(rows_by_cluster, ignore_index=True)
    else:
        candidate_df = pd.DataFrame(columns=candidate_df.columns)

    print("Отзывов-кандидатов после фильтрации кластеров:", len(candidate_df))

    sent_to_model = 0
    scanned_reviews = 0
    skipped_duplicates = 0
    skipped_empty = 0
    saved_examples_before = len(selected_examples)

    class_counts = {label: 0 for label in ALLOWED_LABELS}
    for row in selected_examples:
        labels_counted = row.get("labels_counted", [])
        if isinstance(labels_counted, str):
            labels_counted = parse_labels_str(labels_counted)
        for label in labels_counted:
            if label in class_counts:
                class_counts[label] += 1

    class_bars = {}
    for pos, label in enumerate(ALLOWED_LABELS):
        class_bars[label] = tqdm(
            total=0,
            initial=0,
            desc=label[:35],
            position=pos,
            leave=True,
            bar_format="{desc}: {postfix}",
        )
        class_bars[label].set_postfix({"count": class_counts[label]})

    main_bar = tqdm(
        candidate_df.iterrows(),
        total=len(candidate_df),
        desc="Candidate reviews",
        position=len(ALLOWED_LABELS),
        leave=True,
    )

    try:
        for source_row_idx, row in main_bar:
            if MAX_REVIEWS_TO_SEND_TO_MODEL is not None and sent_to_model >= MAX_REVIEWS_TO_SEND_TO_MODEL:
                print("Достигнут MAX_REVIEWS_TO_SEND_TO_MODEL")
                break

            review_text = str(row.get(TEXT_COL, "")).strip()
            if not review_text or review_text.lower() == "nan":
                skipped_empty += 1
                continue

            current_hash = text_hash(review_text)
            if DEDUP_BY_TEXT and current_hash in existing_hashes:
                skipped_duplicates += 1
                continue

            scanned_reviews += 1
            sent_to_model += 1

            result = classify_review(review_text)
            labels = result.get("labels", [])

            if labels:
                labels_counted = [label for label in labels if label in ALLOWED_LABELS]

                example_row = {
                    TEXT_COL: review_text,
                    "labels": labels,
                    "labels_str": "; ".join(labels),
                    "labels_counted": labels_counted,
                    "labels_counted_str": "; ".join(labels_counted),
                    "confidence": result.get("confidence", 0.0),
                    "evidence": result.get("evidence", ""),
                    "error": result.get("error"),
                    "raw_response": result.get("raw_response", ""),
                    "text_hash": current_hash,
                    "source": "hdbscan_candidate_cluster",
                    "source_row_idx_in_candidate_df": int(source_row_idx),
                    "cluster_id": int(row.get(CLUSTER_COL)),
                    "cluster_probability": float(row.get(PROB_COL, 0.0)),
                    "cluster_title": row.get("cluster_title", ""),
                    "cluster_feature": row.get("cluster_feature", ""),
                    "main_theme": row.get("main_theme", ""),
                    "cluster_matched_labels_str": row.get("matched_labels_str", ""),
                    "cluster_positive_or_negative": row.get("positive_or_negative", ""),
                    "cluster_analysis_confidence": row.get("confidence", 0.0),
                }

                selected_examples.append(example_row)
                existing_hashes.add(current_hash)
                append_example_to_disk(example_row)

                for label in labels_counted:
                    class_counts[label] += 1
                    class_bars[label].set_postfix({"count": class_counts[label]})

            if REQUEST_SLEEP_SECONDS > 0:
                time.sleep(REQUEST_SLEEP_SECONDS)

            main_bar.set_postfix({
                "saved_new": len(selected_examples) - saved_examples_before,
                "sent": sent_to_model,
                "dups": skipped_duplicates,
            })

    finally:
        for bar in class_bars.values():
            bar.close()
        main_bar.close()

    labeled_df = save_full_outputs(selected_examples)
    status_df = build_class_status_df(selected_examples)

    run_stats = {
        "candidate_clusters": len(CANDIDATE_CLUSTER_IDS),
        "candidate_reviews": int(len(candidate_df)),
        "scanned_reviews": int(scanned_reviews),
        "sent_to_model": int(sent_to_model),
        "skipped_duplicates": int(skipped_duplicates),
        "skipped_empty": int(skipped_empty),
        "saved_examples_before": int(saved_examples_before),
        "saved_examples_after": int(len(selected_examples)),
        "saved_new_examples": int(len(selected_examples) - saved_examples_before),
        "class_counts_extra_dataset": dict(zip(status_df["label"], status_df["count_in_extra_dataset"])),
        "output_csv_path": str(OUTPUT_CSV_PATH),
        "final_csv_path": str(FINAL_CSV_PATH),
        "status_csv_path": str(STATUS_CSV_PATH),
    }

    with open(RUN_STATS_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(run_stats, f, ensure_ascii=False, indent=2)

    return labeled_df, status_df, run_stats

## 13. Запуск добора

Для теста можно заранее поставить в настройках, например:
- `MAX_CLUSTERS_TO_PROCESS = 3`
- `MAX_REVIEWS_PER_CLUSTER = 20`
- `MAX_REVIEWS_TO_SEND_TO_MODEL = 50`

Для полного запуска оставь эти параметры `None`.

In [ ]:
 labeled_df, status_df, run_stats = collect_extra_examples_from_candidate_clusters()

print("Статистика запуска:")
print(json.dumps(run_stats, ensure_ascii=False, indent=2))

display(status_df)

## 14. Просмотр результата

In [ ]:
print("Полный CSV:", OUTPUT_CSV_PATH)
print("Финальный CSV:", FINAL_CSV_PATH)
print("Parquet:", OUTPUT_PARQUET_PATH)
print("Статус классов:", STATUS_CSV_PATH)
print("Статистика:", RUN_STATS_JSON_PATH)

if not labeled_df.empty:
    display(labeled_df.tail(20))
else:
    print("Пока не сохранено ни одного примера.")

## 15. Сводка по классам и кластерам

In [ ]:
if not labeled_df.empty:
    exploded = labeled_df.copy()
    exploded["labels_counted_list"] = exploded["labels_counted_str"].apply(parse_labels_str)
    exploded = exploded.explode("labels_counted_list")

    print("Распределение по классам:")
    display(
        exploded.groupby("labels_counted_list")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    print("Топ кластеров по числу сохраненных примеров:")
    display(
        labeled_df.groupby(["cluster_id", "cluster_title", "cluster_matched_labels_str"], dropna=False)
        .size()
        .reset_index(name="saved_count")
        .sort_values("saved_count", ascending=False)
        .head(30)
    )
else:
    print("Нет данных для сводки.")